In [160]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [161]:
import sys
import os
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime
np.NaN=np.nan
import pandas_ta as ta
import importlib
from typing import Tuple
import vectorbt as vbt
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import tqdm
from tabulate import tabulate
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)


In [162]:
START_DATE = "2017-01-01"
END_DATE = "2024-01-01"#datetime.today().strftime('%Y-%m-%d')
INTERVALS = 'data/nifty500/nifty500_weekly_ohlcv.csv'

In [163]:

__all__ = [
    "Strategy",
    "record_monthly_weights",
    "get_rebalancing_dates",
    "backtest",
    "Backtester",
    "plot_strategy_vs_benchmark",
]

universe_files = {
    'Small Cap': 'ind_niftysmallcap100list.csv',
    'Large Cap': 'ind_nifty50list.csv',
    'Mid Cap': 'ind_niftymidcap100list.csv',
    'Micro Cap': 'ind_niftymicrocap250_list.csv',
    'All Cap': 'data/nifty500/ind_nifty500list.csv'
    }


def record_monthly_weights(weights_df, current_date,next_date, top_set):
    """
    On the rebalance date (start of month): weight = 1/N
    On the last trading day of that same month: weight = 0
    """
    if not top_set:
        return

    # 1) Compute weight
    w = 1.0 / len(top_set)

    # 2) Assign 1/N on rebalance date
    if current_date in weights_df.index:
        weights_df.loc[current_date, list(top_set)] = w

    # 3) Find the last trading day in that month
    #    Filter the index to the same year-month, then take the max date
    month = current_date.month
    year  = current_date.year

    # all dates in daily_index that match this month/year
    mask = (
        (weights_df.index.year  == year) &
        (weights_df.index.month == month)
    )
    month_dates = weights_df.index[mask]
    if month_dates.empty:
        return

    # 4) Assign 0 on that last trading day
    weights_df.loc[next_date, list(top_set)] = 0.0

def get_rebalancing_dates(dates, frequency):
    if frequency == "weekly":
        # Every Friday
        return dates[dates.weekday == 4]
    elif frequency == "monthly":
        # Only consider Fridays, then pick the first of each month
        fridays = dates[dates.weekday == 4].sort_values()
        if fridays.empty:
            return pd.DatetimeIndex([])

        # Group by month and pick the earliest Friday in each
        periods = fridays.to_period("M").unique()
        first_fridays = [
            fridays[fridays.to_period("M") == period].min()
            for period in periods
        ]
        return pd.DatetimeIndex(first_fridays)
    else:
        raise ValueError("Frequency must be 'weekly' or 'monthly'.")
    
def backtest(start_date, end_date,z_mean, universe_name, frequency="monthly", 
             initial_capital=100000, number_stocks_active=100, 
             zscore_threshold=0.0,period=48, risk_free_rate=0.04):
    
    # --- 1) Read your daily trading_signals.csv to get the full daily index ---
    # Read with the first column as the index and parse it as dates
    signals_df = pd.read_csv("auxilary/initial_weights.csv",index_col=0,parse_dates=True)


    # This is your universe of all trading days
    daily_index = signals_df.index

    # --- 2) Pre-allocate a weights DataFrame with NaNs on that daily index ---
    weights_df = pd.DataFrame(
        data    = np.nan,
        index   = daily_index,
        columns = signals_df.columns  # all tickers in that file
    )

    z_score_period = period
    z_score_file = 'z_scores_mean.csv' if z_mean else 'z_scores.csv'
    
    # Load the universe from the CSV file
    universe_file = universe_files.get(universe_name)
    if not universe_file:
        raise ValueError(f"Invalid universe name: {universe_name}. Choose from {list(universe_files.keys())}")
    
    universe_df = pd.read_csv(universe_file)
    universe = universe_df.iloc[:, 2].tolist()  # Assuming stock symbols are in the third column
    
    # Load stock prices and z-scores
    stock_prices = pd.read_csv('split_ohlcv_data/all_close.csv', parse_dates=['Date'], index_col='Date')
    z_scores = pd.read_csv(z_score_file, parse_dates=['Date'], index_col='Date')

    # Filter valid stocks
    valid_universe = [stock+".NS" for stock in universe if stock+".NS" in stock_prices.columns]

    # Filter data based on date range
    stock_prices = stock_prices.loc[start_date:end_date, valid_universe]
    z_scores = z_scores.loc[start_date:end_date, valid_universe]

    print('Stock prices and Z-scores loaded.')
    
    portfolio_value = initial_capital
    portfolio_history = []
    dates = z_scores.index
    
    rebalancing_dates = get_rebalancing_dates(pd.to_datetime(dates), frequency)
    
    print(f"Rebalancing dates: {rebalancing_dates[-1]}")
    manual_date = pd.Timestamp('2023-12-29 00:00:00')
    rebalancing_dates = rebalancing_dates.append(pd.DatetimeIndex([manual_date]))
    # rebalancing_dates = rebalancing_dates[rebalancing_dates >= dates[z_score_period]]
    portfolio_history.append({'date': rebalancing_dates[0], 'value': portfolio_value})
    prev_top_stocks = set()

    # Adjust risk-free rate based on rebalancing frequency
    periods_per_year = 52 if frequency == "weekly" else 12
    risk_free_growth_factor = (1 + risk_free_rate) ** (1 / periods_per_year)
    
    for i, current_date in enumerate(rebalancing_dates[:-1]):
        next_date = rebalancing_dates[i + 1]
        
        current_z_scores = z_scores.loc[current_date].dropna()
        
        # Select stocks above the threshold
        top_stocks = current_z_scores[current_z_scores >= zscore_threshold].nlargest(number_stocks_active).index
        
        top_set = set(top_stocks)
        retained_stocks = top_set & prev_top_stocks
        expelled_stocks = prev_top_stocks - top_set
        new_additions = top_set - prev_top_stocks

        print(f"\nRebalancing on {current_date}:")
        print(f"Qualified stocks: {list(top_stocks)}")
        print(f"Retained stocks: {list(retained_stocks)}")
        print(f"Expelled stocks: {list(expelled_stocks)}")
        print(f"New additions: {list(new_additions)}")

        prev_top_stocks = top_set

        # num_selected_stocks = len(top_stocks)
        # allocated_capital = portfolio_value * (num_selected_stocks / number_stocks_active)
        # print(f"Allocated capital: {allocated_capital}")
        # print(f"Unallocated capital: {portfolio_value - allocated_capital}")
        # unallocated_capital = portfolio_value - allocated_capital

        # new_portfolio_value = 0
        # for stock in top_stocks:
        #     try:
        #         buy_price = stock_prices.loc[current_date, stock]
        #         print(f"Buying {stock} at {buy_price}")
        #         sell_price = stock_prices.loc[next_date, stock]
        #         print(f"Selling {stock} at {sell_price}")
        #         new_portfolio_value += (allocated_capital / num_selected_stocks) * (sell_price / buy_price)
        #     except KeyError:
        #         print(f"Skipping {stock} due to missing data.")

        # Apply risk-free growth to unallocated capital
        # portfolio_value = new_portfolio_value + (unallocated_capital * 1) #risk_free_growth_factor)

        # portfolio_history.append({'date': next_date, 'value': portfolio_value})
        record_monthly_weights(weights_df, current_date,next_date, top_set)

    # Log the last rebalancing selection (no return calculated)
    rebalancing_dates.normalize()
    final_date = rebalancing_dates[-1]
    print(final_date)
    final_z_scores = z_scores.loc[final_date].dropna()
    final_top_stocks = final_z_scores[final_z_scores >= zscore_threshold].nlargest(number_stocks_active).index

    final_top_set = set(final_top_stocks)
    final_retained = final_top_set & prev_top_stocks
    final_expelled = prev_top_stocks - final_top_set
    final_new = final_top_set - prev_top_stocks

    print(f"\nFinal Rebalancing on {final_date}:")
    print(f"Qualified stocks: {list(final_top_stocks)}")
    print(f"Retained stocks: {list(final_retained)}")
    print(f"Expelled stocks: {list(final_expelled)}")
    print(f"New additions: {list(final_new)}")
    weights_df.to_csv("auxilary/backtester_weights.csv", index_label="Date")
    # return pd.DataFrame(portfolio_history)



In [164]:
class Backtester:
    def __init__(self, data: pd.DataFrame, initial_value: float):
        self.data = data
        self.initialvalue=initial_value
        self.portfolio_value = initial_value
        self.cash = initial_value
        self.investment = 0.0
        self.current_index = 1
        tickers = data.columns.get_level_values(0).unique()
        self.positions = pd.Series(0, index=tickers)
        self.all_positions = pd.DataFrame(columns=tickers)
        self.tradingState = {}
        self.all_signals = pd.DataFrame(columns=tickers)

    def calculate_positions(self, signal: pd.Series, value, open=True) -> pd.Series:
        if (signal < 0).any():
            raise ValueError(f'For timestamp {self.data.index[self.current_index]}, signal contains negative values: {signal[signal < 0]}')
        if not isinstance(signal, pd.Series):
            raise TypeError(f'For timestamp {self.data.index[self.current_index]}, signal must be a pandas Series, got {type(signal)}')
        if abs(signal).sum() - 1 > 1e-6:
            raise ValueError(f'For timestamp {self.data.index[self.current_index]} the sum of the abs(signals) must not be greater than 1, got {abs(signal).sum()}')

        prices = (
            self.data.xs('Open', level=1, axis=1).iloc[self.current_index]
            if open
            else self.data.xs('Close', level=1, axis=1).iloc[self.current_index]
        )
        prices = prices.reindex(signal.index)
        
        nan_index = signal.isna()
        value -= (self.positions[nan_index]*prices[nan_index]).sum()

        float_shares = (signal.replace(0,np.nan) * value) / prices.replace(0, np.nan)

        float_shares = (
            float_shares
            .replace([np.inf, -np.inf], 0)
            .fillna(0)
        )

        new_positions = pd.Series(0, index=float_shares.index, dtype=int)
        longs  = float_shares > 0
        shorts = float_shares < 0

        new_positions[longs]  = np.floor(float_shares[longs]).astype(int)
        new_positions[shorts] = np.ceil (float_shares[shorts]).astype(int)
        
        new_positions[nan_index] = self.positions[nan_index]

        return new_positions

    def calculate_cash(self, positions: pd.Series, open=True) -> float:
        index = self.current_index
        price = self.data.xs('Open',level=1,axis=1).iloc[index] if open else self.data.xs('Close',level=1,axis=1).iloc[index]
        return self.portfolio_value - (abs(positions) * price).sum()

    def update_investment(self, positions: pd.Series, new_day=False) -> float:
        index = self.current_index
        price1 = self.data.xs('Close',level=1,axis=1).iloc[index-1] if new_day else self.data.xs('Open',level=1,axis=1).iloc[index]
        price2 = self.data.xs('Open',level=1,axis=1).iloc[index] if new_day else self.data.xs('Close',level=1,axis=1).iloc[index]
        return (positions * (price2 - price1)).sum() + self.investment

    def run(self):
        processed_data = Strategy().process_data(self.data)
        self.all_positions.loc[self.data.index[0]] = self.positions
        traderData = 0
        for i in tqdm.tqdm(range(1, len(self.data))):
            self.tradingState = {
                'processed_data': processed_data[:i],
                'investment': self.investment,
                'cash': self.cash,
                'current_timestamp': self.data.index[self.current_index],
                'traderData': traderData,
                'positions': self.positions,
            }
            signal, traderData = Strategy().get_signals(self.tradingState)
            if signal is None:
                raise ValueError(f'For timestamp {self.data.index[self.current_index]}, signal is None')
            self.investment = self.update_investment(self.positions, new_day=True)
            self.portfolio_value = self.investment + self.cash
            self.positions = self.calculate_positions(signal, self.portfolio_value)
            self.cash = self.calculate_cash(self.positions)
            self.investment = self.portfolio_value - self.cash
            self.investment = self.update_investment(self.positions, new_day=False)
            self.portfolio_value = self.investment + self.cash
            self.all_positions.loc[self.data.index[i]] = self.positions
            self.all_signals.loc[self.data.index[i-1]] = signal
            self.current_index += 1

    def vectorbt_run(self):
        open_prices = self.data.xs('Open', level=1, axis=1).loc[self.all_positions.index, self.all_positions.columns]
        close_prices = self.data.xs('Close', level=1, axis=1).loc[self.all_positions.index, self.all_positions.columns]

        order_size = self.all_positions.diff().fillna(0).astype(int)
        order_size = order_size.mask(order_size == 0)

        portfolio = vbt.Portfolio.from_orders(
            close=close_prices,
            size=order_size,
            price=open_prices,
            init_cash=self.initialvalue,
            freq='1D',
            cash_sharing=True,
            call_seq='auto',
            log=True,
        )
        

        benchmark_returns_series = pd.read_csv('benchmark.csv',index_col=0,parse_dates=True)
        benchmark_returns_series = benchmark_returns_series[START_DATE:END_DATE]

        stats_eq = portfolio.stats(settings=dict(benchmark_returns=benchmark_returns_series))
        stats_df = stats_eq.to_frame(name='Value').reset_index()
        stats_df.columns = ['Metric', 'Value']
        
        portfolio.assets().to_csv('results/assets.csv')
        portfolio.orders.records_readable.to_csv('results/log.csv')
        
        df = pd.concat([portfolio.value(),portfolio.asset_value(),portfolio.cash()], axis=1)
        df.columns = ['portfolio', 'investment', 'cash']
        df.to_csv('results/portfolio.csv')

        print(tabulate(stats_df,             headers='keys',
            tablefmt='psql',
            showindex=False,
            floatfmt=".3f"
        ))
        
        return portfolio

In [165]:
def z_score(start_date, end_date, universe, rolling_threshold=None, period=None, mean_period=None):
    # Load the appropriate stock list based on the universe
    if universe == 'Small Cap':
        symbols_df = pd.read_csv('data/niftysmallcap100/ind_niftysmallcap100list.csv')
    elif universe == 'Large Cap':
        symbols_df = pd.read_csv('data/nifty50/ind_nifty50list.csv')
    elif universe == 'Mid Cap':
        symbols_df = pd.read_csv('data/niftymidcap100/ind_niftymidcap100list.csv')
    elif universe == 'Micro Cap':
        symbols_df = pd.read_csv('data/niftymicrocap250/ind_niftymicrocap250_list.csv')
    elif universe == 'All Cap':
        symbols_df = pd.read_csv('data/nifty500/ind_nifty500list.csv')
    else:
        raise ValueError("Invalid universe selected.")

    symbols = symbols_df['Symbol'].tolist()
    
    # Load weekly closing prices
    weekly_close = pd.read_csv('split_ohlcv_data/all_close.csv', parse_dates=['Date'], index_col='Date')
    
    # Filter symbols that are present in the price data
    available_symbols = [symbol + ".NS" for symbol in symbols if symbol + ".NS" in weekly_close.columns]
    if not available_symbols:
        raise ValueError("None of the selected symbols are present in all_close.csv")

    # Slice data based on start and end date
    weekly_close = weekly_close.loc[start_date:end_date, available_symbols]

    # Calculate weekly log returns
    weekly_returns = np.log(weekly_close / weekly_close.shift(1)).dropna()

    # Compute cross-sectional z-scores for every week
    z_scores_df = (weekly_returns.sub(weekly_returns.mean(axis=1), axis=0)
                                .div(weekly_returns.std(axis=1), axis=0))

    z_scores_df.to_csv("z_scores.csv", index_label="Date")

    # Optional: mean z-score can just be the same since only 1 row exists
    z_scores_mean_df = z_scores_df.copy()
    z_scores_mean_df.to_csv("z_scores_mean.csv", index_label="Date")

    return z_scores_df, z_scores_mean_df


In [166]:
z_scores_df, z_scores_mean_df = z_score(START_DATE, END_DATE, 'All Cap', 0, 24, 16)
z_mean = False
backtest('2016-07-01', '2025-07-11', z_mean, 'All Cap', 'monthly',100000, 50, 0.0, 24)

Stock prices and Z-scores loaded.
Rebalancing dates: 2023-12-01 00:00:00

Rebalancing on 2017-01-06 00:00:00:
Qualified stocks: ['PGEL.NS', 'USHAMART.NS', 'KEI.NS', 'DCMSHRIRAM.NS', 'SWANENERGY.NS', 'JSWENERGY.NS', 'WOCKPHARMA.NS', 'GRAPHITE.NS', 'NATCOPHARM.NS', 'RCF.NS', 'ACE.NS', 'NAVA.NS', 'BALRAMCHIN.NS', 'CHAMBLFERT.NS', 'ZYDUSLIFE.NS', 'JWL.NS', 'HSCL.NS', 'FORTIS.NS', 'GPIL.NS', 'YESBANK.NS', 'INDIANB.NS', 'CUB.NS', 'BERGEPAINT.NS', 'KOTAKBANK.NS', 'ASAHIINDIA.NS', 'BBTC.NS', 'SIEMENS.NS', '3MINDIA.NS', 'TECHNOE.NS', 'EICHERMOT.NS', 'GPPL.NS', 'OIL.NS', 'GODREJIND.NS', 'ASHOKLEY.NS', 'BAJAJHLDNG.NS', 'BANKBARODA.NS', 'CANBK.NS', 'NAUKRI.NS', 'INDUSINDBK.NS', 'ONGC.NS', 'SHRIRAMFIN.NS', 'ASIANPAINT.NS', 'BIOCON.NS', 'RPOWER.NS', 'CANFINHOME.NS', 'TORNTPHARM.NS', 'CERA.NS', 'ATUL.NS', 'SARDAEN.NS', 'ICICIPRULI.NS']
Retained stocks: []
Expelled stocks: []
New additions: ['ZYDUSLIFE.NS', 'HSCL.NS', 'USHAMART.NS', 'RPOWER.NS', 'BIOCON.NS', 'ASHOKLEY.NS', 'TORNTPHARM.NS', 'BANKBARODA

In [167]:
class Strategy():
    
   signalsData = signalsData = pd.read_csv(
        'auxilary/backtester_weights.csv', #replace with your file path
        na_values=['nan', 'NaN', ''],
        keep_default_na=True
    )
   signalsData.set_index(signalsData.columns[0], inplace=True)
   
   def process_data(self, data) -> pd.DataFrame:
      return data

   def get_signals(self, tradingState: dict) -> Tuple[list, str]:

      signal = Strategy.signalsData.iloc[tradingState['traderData']]
      tickers = signal.index.tolist()
      signal = pd.Series(signal.values, index=tickers)
      traderData = tradingState['traderData'] + 1
   
      return signal, traderData

In [168]:
data = pd.read_csv(
    'data/nifty500/nifty500_daily_ohlcv.csv',
    index_col=0, header=[0,1], parse_dates=True
)

# tickers = data.columns.get_level_values(0).unique()[:500]
# data = data.loc[:, data.columns.get_level_values(0).isin(tickers)]
initial_value = 100000.0
backtester = Backtester(data, initial_value)
backtester.run()
backtester.all_signals.to_csv('results/signals.csv')
print(backtester.portfolio_value)
pf = backtester.vectorbt_run()


100%|██████████| 1728/1728 [00:07<00:00, 218.27it/s]


585400.5795682073
+----------------------------+----------------------------+
| Metric                     | Value                      |
|----------------------------+----------------------------|
| Start                      | 2017-01-02 00:00:00        |
| End                        | 2023-12-29 00:00:00        |
| Period                     | 1729 days 00:00:00         |
| Start Value                | 100000.0                   |
| End Value                  | 585400.5795682073          |
| Total Return [%]           | 485.4005795682073          |
| Benchmark Return [%]       | 426.536104499507           |
| Max Gross Exposure [%]     | 97.51315183552809          |
| Total Fees Paid            | 0.0                        |
| Max Drawdown [%]           | 46.93956128858441          |
| Max Drawdown Duration      | 713 days 00:00:00          |
| Total Trades               | 3460                       |
| Total Closed Trades        | 3411                       |
| Total Open Trades   

In [169]:
import plotly.graph_objects as go

stats_eq = pf.stats()
eq_curve = pf.value()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=eq_curve.index,
    y=eq_curve.values,
    mode='lines',
    name='Equity Curve',
    line=dict(color='green', width=3)
))
fig.update_layout(
    title='Portfolio Equity Curve',
    xaxis_title='Date',
    yaxis_title='Portfolio Value',
    template='plotly_white'
)
fig.show()


In [170]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

returns = eq_curve.pct_change().fillna(0)
cum_max = eq_curve.cummax()
drawdown = (eq_curve - cum_max) / cum_max
mean_ret = returns.mean()
median_ret = returns.median()

fig = make_subplots(rows=1, cols=2, subplot_titles=('Daily Returns', 'Drawdown Curve'))
fig.add_trace(
    go.Histogram(
        x=returns,
        nbinsx=50,
        marker_color='orange',
        marker_line_color='white',
        marker_line_width=1,
        opacity=0.8,
        name='Returns'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x=drawdown.index,
        y=drawdown.values,
        mode='lines',
        line=dict(color='red', width=2),
        name='Drawdown'
    ),
    row=1, col=2
)

fig.update_xaxes(
    title_text='Return',
    row=1, col=1,
    nticks=20,
    showgrid=True
)
fig.update_yaxes(
    title_text='Frequency',
    row=1, col=1,
    showgrid=True
)

fig.update_xaxes(
    title_text='Date',
    row=1, col=2,
    showgrid=False
)
fig.update_yaxes(
    title_text='Drawdown',
    row=1, col=2,
    showgrid=True
)

fig.update_layout(
    title_text='Returns Distribution & Drawdown',
    bargap=0.1,               
    template='plotly_white',
    showlegend=False,
    width=900,
    height=400
)

fig.show()
